#### Импорт библиотек и создание папки для хранения графиков и изображений

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path.cwd() / 'fuzzy'

#### Реализации функций принадлежности для нечётких множеств и вспомогательную функцию для их вызова

In [2]:
def trimf(x, abc):
    a, b, c = abc
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    if b != a:
        y = np.maximum(y, np.minimum((x-a)/(b-a), 1))
    else:
        y = np.maximum(y, (x <= b).astype(float))
    if c != b:
        y = np.minimum(y, np.maximum((c-x)/(c-b), 0))
    else:
        y = np.minimum(y, (x >= b).astype(float))
    return np.clip(y, 0, 1)

def trapmf(x, abcd):
    a, b, c, d = abcd
    x = np.asarray(x, dtype=float)

    if b != a:
        left = (x - a) / (b - a)
    else:
        left = (x >= b).astype(float)

    if d != c:
        right = (d - x) / (d - c)
    else:
        right = (x <= c).astype(float)

    y = np.minimum(left, right)
    return np.clip(y, 0, 1)

def mf(spec, x):
    kind, pts = spec

    if kind == 'tri':
        result = trimf(x, pts)
    else:
        result = trapmf(x, pts)

    if np.isscalar(x):
        return float(result)

    return result

#### Задание областей определения и лингвистических терм переменных

In [32]:
error_u = np.linspace(-20, 20, 401)       # target - current, C
rate_u = np.linspace(-2, 2, 401)          # C/s
angle_u = np.linspace(0, 90, 901)         # degrees

ERROR = {
    'very_hot': ('trap', (-20, -20, -14, -4)),
    'hot': ('tri', (-14, -4, 0)),
    'comfortable': ('tri', (-4, 0, 4)),
    'cold': ('tri', (0, 4, 14)),
    'very_cold': ('trap', (4, 14, 20, 20)),
}
RATE = {
    'cooling': ('trap', (-2, -2, -0.5, 0)),
    'stable': ('tri', (-0.5, 0, 0.5)),
    'warming': ('trap', (0, 0.5, 2, 2)),
}
ANGLE = {
    'closed': ('trap', (0, 0, 5, 25)),
    'low': ('tri', (5, 25, 45)),
    'medium': ('tri', (25, 45, 65)),
    'high': ('tri', (45, 65, 85)),
    'max': ('trap', (65, 85, 90, 90)),
}

#### Задание правил системы

In [4]:
# Rule base: (error_term, rate_term, hot_angle_term, cold_angle_term)
RULES = [
    # Вода очень горячая
    ('very_hot', 'warming', 'closed', 'max'),
    ('very_hot', 'stable',  'closed', 'high'),
    ('very_hot', 'cooling', 'closed', 'medium'),

    # Вода горячая
    ('hot', 'warming', 'low', 'max'),
    ('hot', 'stable',  'low', 'high'),
    ('hot', 'cooling', 'low', 'medium'),

    # Вода комфортная
    ('comfortable', 'warming', 'low',    'medium'),
    ('comfortable', 'stable',  'medium', 'medium'),
    ('comfortable', 'cooling', 'medium', 'low'),

    # Вода холодная
    ('cold', 'warming', 'medium', 'low'),
    ('cold', 'stable',  'high', 'low'),
    ('cold', 'cooling', 'max',   'low'),

    # Вода очень холодная
    ('very_cold', 'warming', 'medium', 'closed'),
    ('very_cold', 'stable',  'high',   'closed'),
    ('very_cold', 'cooling', 'max',    'closed'),
]

#### Реализации ядра нечёткой системы: деффазификация и функция нечёткого вывода с нормализацией потока

In [5]:
def centroid(xs, ys):
    area = np.trapz(ys, xs)
    if area <= 1e-9:
        return 45.0
    return float(np.trapz(xs * ys, xs) / area)

FLOW_TOTAL = 90.0  # постоянный суммарный напор

def infer(error, rate):
    hot_agg = np.zeros_like(angle_u)
    cold_agg = np.zeros_like(angle_u)

    for e_term, r_term, hot_term, cold_term in RULES:
        alpha = min(mf(ERROR[e_term], error), mf(RATE[r_term], rate))

        if alpha <= 0:
            continue

        hot_mf = np.minimum(alpha, mf(ANGLE[hot_term], angle_u))
        cold_mf = np.minimum(alpha, mf(ANGLE[cold_term], angle_u))

        hot_agg = np.maximum(hot_agg, hot_mf)
        cold_agg = np.maximum(cold_agg, cold_mf)

    hot_angle = centroid(angle_u, hot_agg)
    cold_angle = centroid(angle_u, cold_agg)

    total = hot_angle + cold_angle

    if total <= 1e-9:
        return FLOW_TOTAL / 2, FLOW_TOTAL / 2

    hot_angle = hot_angle / total * FLOW_TOTAL
    cold_angle = cold_angle / total * FLOW_TOTAL

    return hot_angle, cold_angle

#### Функция для построения графиков функций принадлежности

In [6]:
def plot_mfs(universe, terms, title, path, xlabel):
    plt.figure(figsize=(8, 4.2))
    
    for name, spec in terms.items():
        plt.plot(universe, mf(spec, universe), label=name, linewidth=2)
    
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Степень принадлежности')
    plt.ylim(-0.05, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(loc='best', fontsize=8)
    plt.tight_layout()
    
    plt.savefig(path, dpi=180)
    plt.close()

#### Функция для построения поверхностей вывода

In [7]:
def build_surfaces():
    e_vals = np.linspace(-20, 20, 61)
    r_vals = np.linspace(-2, 2, 61)
    E, R = np.meshgrid(e_vals, r_vals)

    H = np.zeros_like(E)
    C = np.zeros_like(E)

    for i in range(E.shape[0]):
        for j in range(E.shape[1]):
            H[i, j], C[i, j] = infer(E[i, j], R[i, j])

    for Z, name, title in [
        (H, 'surface_hot.png', 'Поверхность вывода: горячий кран'),
        (C, 'surface_cold.png', 'Поверхность вывода: холодный кран')
    ]:
        fig = plt.figure(figsize=(8, 5.4))
        ax = fig.add_subplot(111, projection='3d')
        surf = ax.plot_surface(E, R, Z, cmap='viridis', linewidth=0,
                               antialiased=True, alpha=0.95)

        ax.set_title(title)
        ax.set_xlabel('Ошибка, °C')
        ax.set_ylabel('Скорость, °C/с')
        ax.set_zlabel('Угол, °')

        fig.colorbar(surf, shrink=0.65, aspect=14)
        plt.tight_layout()
        plt.savefig(BASE_DIR / name, dpi=180)
        plt.close()

#### Функция для симуляции системы и построения графиков динамики

In [8]:
def simulate(initial_temp, steps=80, target=38.0, hot_temp=65.0, cold_temp=11.0):
    temp = initial_temp
    rate = 0.0

    hot_angle = 45.0
    cold_angle = 45.0

    temps, rates, hot_angles, cold_angles = [], [], [], []

    for _ in range(steps):
        error = target - temp

        desired_hot, desired_cold = infer(error, rate)

        # сглаживаем изменение положения кранов
        beta = 0.15
        hot_angle = hot_angle + beta * (desired_hot - hot_angle)
        cold_angle = cold_angle + beta * (desired_cold - cold_angle)

        total = hot_angle + cold_angle

        if total < 1e-6:
            mixed = target
        else:
            mixed = (hot_angle * hot_temp + cold_angle * cold_temp) / total

        # сглаживаем изменение температуры
        new_temp = temp + 0.08 * (mixed - temp)

        # сглаживаем скорость изменения температуры
        new_rate = np.clip(0.7 * rate + 0.3 * (new_temp - temp), -2, 2)

        temps.append(temp)
        rates.append(rate)
        hot_angles.append(hot_angle)
        cold_angles.append(cold_angle)

        temp, rate = new_temp, new_rate

    return np.array(temps), np.array(rates), np.array(hot_angles), np.array(cold_angles)

def plot_sim(initial_temp, path):
    temps, rates, hot_angles, cold_angles = simulate(initial_temp)
    x = np.arange(len(temps))

    fig, ax1 = plt.subplots(figsize=(8.5, 4.8))

    ax1.plot(x, temps, linewidth=2, label='Температура воды')
    ax1.axhspan(36, 40, alpha=0.15, label='Комфортная зона 36-40°C')
    ax1.set_xlabel('Шаг моделирования')
    ax1.set_ylabel('Температура, °C')
    ax1.grid(True, alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(x, hot_angles, linestyle='--', label='Горячий кран', color='orange')
    ax2.plot(x, cold_angles, linestyle=':', label='Холодный кран', color='steelblue')
    ax2.set_ylabel('Угол открытия, °')

    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='best', fontsize=8)

    plt.title(f'Динамика системы при начальной температуре {initial_temp}°C')
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()

#### Основной блок действий

In [34]:
plot_mfs(error_u, ERROR, 'Входная переменная error', BASE_DIR / 'error_mf.png', 'target - current, °C')
plot_mfs(rate_u, RATE, 'Входная переменная rate', BASE_DIR / 'rate_mf.png', 'Скорость изменения температуры, °C/с')
plot_mfs(angle_u, ANGLE, 'Выходная переменная angle', BASE_DIR / 'angle_mf.png', 'Угол открытия крана, °')

build_surfaces()

for temp in [25, 50, 38]:
    plot_sim(temp, BASE_DIR / f'sim_{temp}.png')
    
examples = [(25, 0), (50, 0.5), (38, 0), (33, -0.3), (44, 0.2)]
print('temp, rate, error, hot_angle, cold_angle')

for cur, rate in examples:
    e = 38 - cur
    h, c = infer(e, rate)
    print(f'{cur:5.1f} {rate:5.2f} {e:6.2f} {h:8.2f} {c:8.2f}')


temp, rate, error, hot_angle, cold_angle
 25.0  0.00  13.00    76.06    13.94
 50.0  0.50 -12.00    13.73    76.27
 38.0  0.00   0.00    45.00    45.00
 33.0 -0.30   5.00    67.20    22.80
 44.0  0.20  -6.00    22.87    67.13
